# Text-to-Video Generation with CogVideoX-2b

A minimal, portfolio-ready pipeline for generating short video clips from text
prompts using [CogVideoX-2b](https://huggingface.co/THUDM/CogVideoX-2b), an
open-source (Apache 2.0) text-to-video diffusion model from THUDM.

**Why this model:** CogVideoX-2b is one of the lightest open video diffusion
models available — it runs in as little as ~4GB of VRAM with fp16 precision,
which comfortably fits the free T4 GPU (16GB VRAM) offered by Google Colab and
Kaggle Notebooks. No local GPU required.

**Run this notebook on:**
- [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alirezayaned/AI-Video-Generation/blob/main/notebooks/cogvideox_text_to_video.ipynb)
- Kaggle: upload this notebook and enable a GPU accelerator (Settings > Accelerator > GPU T4 x2)

## 1. Check your runtime

Make sure you've selected a GPU runtime before running anything below:
- **Colab:** Runtime > Change runtime type > Hardware accelerator > GPU
- **Kaggle:** Notebook Settings (right sidebar) > Accelerator > GPU T4 x2

Free-tier GPUs are not guaranteed to be available on demand, and sessions are
capped (Colab: ~12 hours, Kaggle: ~30 GPU-hours/week, 12 hours/session) — if
`torch.cuda.is_available()` returns `False` below, you likely need to wait and
retry, or switch platforms.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram_gb:.1f} GB")
else:
    print("No GPU detected. Enable a GPU accelerator and restart the runtime before continuing.")


**If you've already hit a CUDA out-of-memory error in this session, restart the kernel/runtime now** (Kaggle: session menu → restart; Colab: Runtime → Restart session) **before continuing** — re-running cells alone does not reliably free GPU memory from a crashed run. The cell below shows how much GPU memory is already in use; if it reports more than a few hundred MB before you've loaded anything, that's leftover from an earlier crash and you need to restart, not just re-run.

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f"GPU memory allocated: {allocated:.2f} GB")
    print(f"GPU memory reserved:  {reserved:.2f} GB")
    if reserved > 0.5:
        print("\nWarning: GPU memory is already in use before loading any model.")
        print("This usually means a previous crash left memory stranded -- restart the kernel/runtime and run from the top.")


## 2. Install dependencies

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg


## 3. Load the model

`THUDM/CogVideoX-2b` downloads in fp16 (~13.8GB of weights, dominated by its
T5-XXL text encoder). Loaded naively that needs ~33GB of VRAM at inference
time -- more than a free T4 has -- so the memory-saving options below are
enabled by default, not optional.

In [ ]:
from diffusers import CogVideoXPipeline

MODEL_ID = "THUDM/CogVideoX-2b"

pipe = CogVideoXPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16)

# Required on free-tier GPUs (T4, ~15GB VRAM). CogVideoX-2b needs ~33GB with no
# offloading at all, and still ~19GB with cpu offload alone -- both exceed a T4.
# cpu offload + VAE tiling together bring it down to ~11GB, which fits with
# headroom to spare. Don't also call pipe.to("cuda") -- enable_model_cpu_offload
# manages device placement itself, and calling .to() on top of it defeats the
# whole point (and is what caused the OOM in the first version of this notebook).
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()

print("Model loaded (memory-optimized for free-tier GPUs; ~11GB VRAM).")

# If you're running on a larger GPU (A100/L4 40GB+, e.g. Colab Pro) and want
# faster generation instead, replace the three lines above with:
#   pipe = pipe.to("cuda")
# which skips offloading entirely at the cost of needing ~33GB VRAM.


## 4. Generate a video from a text prompt

In [ ]:
import os
from diffusers.utils import export_to_video

os.makedirs("outputs", exist_ok=True)

def generate_video(
    prompt: str,
    output_path: str = "outputs/output.mp4",
    num_frames: int = 49,      # CogVideoX-2b is trained on 49-frame clips (~6s at 8fps)
    num_inference_steps: int = 50,
    guidance_scale: float = 6.0,
    seed: int = 42,
    fps: int = 8,
) -> str:
    """Generate a short video clip from a text prompt and save it to disk."""
    generator = torch.Generator(device="cuda").manual_seed(seed)
    frames = pipe(
        prompt=prompt,
        num_frames=num_frames,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        generator=generator,
    ).frames[0]
    export_to_video(frames, output_path, fps=fps)
    return output_path


In [ ]:
prompt = (
    "A golden retriever running through a sunlit meadow, slow motion, "
    "cinematic lighting, shallow depth of field, grass blowing in the wind"
)

video_path = generate_video(prompt, output_path="outputs/sample_01.mp4")
print(f"Saved to {video_path}")


## 5. Preview the result inline

In [ ]:
from IPython.display import Video

Video(video_path, embed=True, width=480)


## 6. (Optional) Convert to GIF for your README

GitHub renders GIFs inline but not MP4s, so a short GIF is the easiest way to
show off a sample result in your repo's README.

In [ ]:
!pip install -q moviepy
from moviepy.editor import VideoFileClip

clip = VideoFileClip(video_path)
gif_path = video_path.replace(".mp4", ".gif")
clip.write_gif(gif_path, fps=8)
print(f"Saved GIF to {gif_path}")


## 7. Optional: add an AI-generated soundtrack

CogVideoX only generates silent video — there's no audio track to extract,
and no setting that adds one. The practical way to get sound onto a clip is a
separate step: generate a short audio clip with a different, much smaller
model, then mux it onto the video.

This uses [MusicGen-small](https://huggingface.co/facebook/musicgen-small)
(Meta, ~2GB VRAM) — a text-to-music model. It generates music/ambience
matching the *mood* you describe in a separate prompt; it doesn't see what's
happening in the video frame-by-frame, so this gives you a soundtrack, not
synced sound effects. For narration instead of music, swap this step for a
text-to-speech model (e.g. Bark, Coqui TTS) using the same mux pattern below.

We free the CogVideoX pipeline from GPU memory first — the free tier doesn't
have room for both models loaded at once.

In [ ]:
import gc

del pipe
gc.collect()
torch.cuda.empty_cache()

from transformers import AutoProcessor, MusicgenForConditionalGeneration

music_processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
music_model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small").to("cuda")
print("MusicGen loaded.")


In [ ]:
import os
import scipy.io.wavfile
from moviepy.editor import VideoFileClip, AudioFileClip

def add_soundtrack(video_path, music_prompt, output_path=None, extra_seconds=1.0, fps=8):
    """Generate a short music clip and mux it onto an existing (silent) video.

    Produces exactly one combined output file -- the intermediate .wav is
    deleted once the mux succeeds.
    """
    video_clip = VideoFileClip(video_path)
    duration = video_clip.duration

    # MusicGen generates audio tokens at 50/sec; pad slightly so we never run short.
    tokens_needed = int((duration + extra_seconds) * 50)

    inputs = music_processor(text=[music_prompt], padding=True, return_tensors="pt").to("cuda")
    audio_values = music_model.generate(**inputs, max_new_tokens=tokens_needed)

    audio_path = video_path.rsplit(".", 1)[0] + "_music.wav"
    sampling_rate = music_model.config.audio_encoder.sampling_rate
    scipy.io.wavfile.write(audio_path, rate=sampling_rate, data=audio_values[0, 0].cpu().numpy())

    audio_clip = AudioFileClip(audio_path).subclip(0, duration)
    final_clip = video_clip.set_audio(audio_clip)

    if output_path is None:
        output_path = video_path.rsplit(".", 1)[0] + "_with_sound.mp4"

    # moviepy 1.0.3 has a known bug where passing fps= as a keyword argument
    # to write_videofile() can silently get dropped (github.com/Zulko/moviepy/issues/2158),
    # falling back to clip.fps -- which is also None for some imageio-written
    # mp4s. Setting the attribute directly bypasses the broken keyword path.
    final_clip.fps = fps
    final_clip.write_videofile(output_path, codec="libx264", audio_codec="aac", logger=None)

    video_clip.close()
    audio_clip.close()
    os.remove(audio_path)  # keep only the single combined output file
    return output_path


video_with_sound = add_soundtrack(
    video_path,
    music_prompt="gentle acoustic guitar, warm and nostalgic, outdoor summer afternoon",
)
print(f"Saved to {video_with_sound}")


In [ ]:
from IPython.display import Video

Video(video_with_sound, embed=True, width=480)


## Notes & next steps

- **Resolution:** CogVideoX-2b defaults to 720x480, which is what it was trained
  on — pushing resolution higher tends to hurt quality more than it helps.
- **Quota awareness:** each generation above takes a few minutes on a T4. Batch
  your prompts in a list and loop `generate_video()` rather than re-running
  cells one at a time, so you don't waste session time re-loading the model.
- **Going further:** once this works, try swapping in
  [HunyuanVideo-1.5](https://huggingface.co/tencent/HunyuanVideo-1.5) (via the
  [Wan2GP](https://github.com/deepbeepmeep/Wan2GP) runner, ~6GB VRAM) for
  noticeably higher quality output on the same free GPU tier.
- **Turning this into a demo:** wrap `generate_video()` in a small
  [Gradio](https://www.gradio.app/) UI and deploy it to a free
  [Hugging Face Space](https://huggingface.co/spaces) so people can try your
  project without running the notebook themselves.
